# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---


## Setup — Run this cell first


In [3]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('https://raw.githubusercontent.com/Mani-Teja9/data-viz-class-material-w2026/main/data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
df.head()


Dataset: 63 countries, 9 columns


,Country,Region,Happiness_Score,GDP,Social_Support,Life_Expectancy,Freedom,Generosity,Corruption
0,Finland,Western Europe,7.804,10.775,0.954,71.9,0.949,0.142,0.179
1,Denmark,Western Europe,7.586,10.933,0.954,72.7,0.931,0.168,0.234
2,Iceland,Western Europe,7.525,10.878,0.983,72.5,0.961,0.260,0.150
3,Israel,Middle East and North Africa,7.473,10.527,0.916,72.4,0.903,0.149,0.826
4,Netherlands,Western Europe,7.464,11.015,0.939,72.4,0.879,0.240,0.296


In [4]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [5]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort for horizontal bar

print(region_avg)

# Step 2: Build your chart
# Insight: the story is the SPREAD between regions, so we grey out the middle and
# colour only the two ends. Zero baseline, sorted, horizontal.
top_region   = region_avg.iloc[-1]['Region']
worst_region = region_avg.iloc[0]['Region']
top_val      = region_avg.iloc[-1]['Happiness_Score']
worst_val    = region_avg.iloc[0]['Happiness_Score']
gap          = top_val - worst_val

region_avg['Highlight'] = np.select(
    [region_avg['Region'] == top_region, region_avg['Region'] == worst_region],
    ['Top', 'Bottom'], default='Other')

fig = px.bar(
    data_frame=region_avg,
    x='Happiness_Score', y='Region',
    orientation='h',                                   # Rule 3
    color='Highlight',
    color_discrete_map={'Top': 'darkblue', 'Bottom': 'crimson', 'Other': 'lightgrey'},
    text='Happiness_Score',
    title=(f'{top_region} averages {top_val:.1f} on happiness, {worst_region} only '
           f'{worst_val:.1f} — a {gap:.1f}-point gap between regions'),   # Rule 4
    labels={'Happiness_Score': 'Average happiness score (0–10 scale)', 'Region': ''},
    template='plotly_white',
    height=500,
)
fig.update_traces(texttemplate='%{x:.1f}', textposition='outside', cliponaxis=False)
fig.update_layout(
    xaxis=dict(range=[0, 10], showgrid=False),         # Rule 1: zero baseline
    font=dict(family='Arial', size=13),
    showlegend=False,
    margin=dict(l=10, r=30, t=70, b=40),
)
fig.show()




                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [6]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
top_mean    = top8['Happiness_Score'].mean()
bottom_mean = bottom8['Happiness_Score'].mean()
gap         = top_mean - bottom_mean

fig = px.bar(
    data_frame=combined,                               # already sorted ascending
    x='Happiness_Score', y='Country',
    orientation='h',                                   # Rule 3
    color='Group',
    color_discrete_map={'Top 8': 'darkblue', 'Bottom 8': 'crimson'},
    text='Happiness_Score',
    hover_data={'Region': True, 'Group': False},
    title=(f'The 8 happiest countries average {top_mean:.1f}, the 8 least happy '
           f'only {bottom_mean:.1f} — a {gap:.1f}-point gap'),         # Rule 4
    labels={'Happiness_Score': 'Happiness score (0–10 scale)', 'Country': ''},
    template='plotly_white',
    height=650,
)
fig.update_traces(texttemplate='%{x:.1f}', textposition='outside', cliponaxis=False)

# Stretch: global (sample) average reference line
fig.add_vline(
    x=global_avg, line_dash='dash', line_color='slategrey', line_width=1.5,
    annotation_text=f'Sample average ({len(df)} countries): {global_avg:.1f}',
    annotation_position='top', annotation_font=dict(size=12, color='slategrey'))

# Visual separator between the two groups (16 bars -> boundary between index 7 and 8)
fig.add_hline(y=7.5, line_color='black', line_width=1)

# Group labels on the right-hand side
fig.add_annotation(x=9.4, y=11.5, text='<b>Top 8</b>', showarrow=False,
                   font=dict(color='darkblue', size=14))
fig.add_annotation(x=9.4, y=3.5, text='<b>Bottom 8</b>', showarrow=False,
                   font=dict(color='crimson', size=14))

fig.update_layout(
    xaxis=dict(range=[0, 10], showgrid=False),         # Rule 1: zero baseline
    font=dict(family='Arial', size=13),
    showlegend=False,
    margin=dict(l=10, r=30, t=90, b=40),
)
fig.show()




Global average: 5.81


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [7]:
# Stretch goal — grouped bar chart
#
# Two fixes to the task text, otherwise the chart would be wrong:
#  1. The column is called 'GDP' (not 'GDP_per_capita'), AND it is log GDP per capita
#     (values ~6-11), while Freedom is 0-1. Putting them on one axis would make
#     Freedom look like zero. So we compare two 0-1 factors instead:
#     Social_Support vs Freedom.
#  2. The region is called 'Latin America and Caribbean' in this dataset.
regions = ['Western Europe', 'Latin America and Caribbean', 'East Asia',
           'Sub-Saharan Africa', 'South Asia']
factors = ['Social_Support', 'Freedom']

grp = (df[df['Region'].isin(regions)]
       .groupby('Region')[factors].mean()
       .sort_values('Social_Support')                  # Rule 2
       .reset_index())

higher = grp[grp['Social_Support'] > grp['Freedom']]['Region'].tolist()
exceptions = grp[grp['Social_Support'] <= grp['Freedom']]['Region'].tolist()

long = grp.melt(id_vars='Region', var_name='Factor', value_name='Score')
long['Factor'] = long['Factor'].str.replace('_', ' ')

fig = px.bar(
    long, x='Score', y='Region',
    orientation='h',                                   # Rule 3
    color='Factor', barmode='group',
    color_discrete_map={'Social Support': 'darkblue', 'Freedom': '#E69F00'},
    text='Score',
    title=(f'Social support beats freedom in {len(higher)} of {len(regions)} regions — '
           f'only {" and ".join(exceptions)} {"is" if len(exceptions)==1 else "are"} the exception'),   # Rule 4
    category_orders={'Factor': ['Freedom', 'Social Support']},
    labels={'Score': 'Average score (0–1 scale)', 'Region': '', 'Factor': ''},
    template='plotly_white',
    height=550,
)
fig.update_traces(texttemplate='%{x:.2f}', textposition='outside', cliponaxis=False)
fig.update_layout(
    xaxis=dict(range=[0, 1.1], showgrid=False),        # Rule 1: zero baseline
    font=dict(family='Arial', size=13),
    legend=dict(orientation='h', y=1.0, yanchor='bottom', x=0, traceorder='reversed'),
    margin=dict(l=10, r=30, t=120, b=40),
)
fig.show()
